# Red residual

(Práctica, 4 puntos)

En la práctica anterior implementamos una red convolucional relativamente poco profunda (LeNet-5). Al apilar más capas convolucionales esperaríamos mejor desempeño, pero en la práctica una CNN *plana* muy profunda suele **degradarse**: la pérdida de entrenamiento deja de bajar e incluso empeora. No es solo sobreajuste; el descenso por gradiente tiene cada vez más dificultad para propagar la señal a las primeras capas (*vanishing gradient*).

Una **red residual** (ResNet) ataca ese problema con *conexiones de salto* (*skip connections*). En lugar de aprender una transformación $H(x)$ desde cero, un bloque residual aprende el **residuo** $F(x)$ respecto a la identidad:

$$
H(x) = F(x) + x
$$

Si la identidad ya es una buena solución, la red puede llevar $F(x)$ cerca de cero. En el pase hacia atrás el gradiente también fluye por el atajo, lo que facilita entrenar redes más profundas.

```
x ──────────────┐
  conv → ReLU   |
  conv → BN     │
        +  ←────┘   suma residual
       ReLU
```

En esta práctica compararás **la misma profundidad** con y sin atajos, sobre un conjunto más exigente que MNIST.

Instrucciones: completa el código marcado con `TODO:`.

@juan1rving

[1] He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.

## Conjunto de datos: CIFAR-10

En esta práctica usaremos [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html): 50,000 imágenes de entrenamiento y 10,000 de prueba, a color, de $32 \times 32$. Hay diez clases de objetos (avión, automóvil, pájaro, ...). Frente a MNIST o Fashion-MNIST:

- las imágenes son RGB (3 canales) y más variadas;
- las clases se confunden con más facilidad;

Esa es la comparación que hicieron He et al. en CIFAR-10 [1].

[2] Krizhevsky, A. (2009). Learning multiple layers of features from tiny images.

In [ ]:
# Cargamos paquetes necesarios

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import numpy as np
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

In [ ]:
# Hiper-parámetros
# NOTE: puedes modificarlos, pero no es necesario para observar la diferencia.

batch_size = 128
tasa_de_aprendizaje = 0.001
epocas = 5
semilla = 42

torch.manual_seed(semilla)
np.random.seed(semilla)

In [ ]:
# Definimos una transformación de los datos
# Media y desviación estándar de CIFAR-10
media = (0.4914, 0.4822, 0.4465)
desv = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(media, desv)
])

# Descargamos el conjunto de entrenamiento y de prueba
trainset = datasets.CIFAR10('data/CIFAR10/', download=True, train=True, transform=transform)
testset = datasets.CIFAR10('data/CIFAR10/', download=True, train=False, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

print('Train data, number of images: ', len(trainset))
print('Test data, number of images: ', len(testset))

classes = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
           'perro', 'rana', 'caballo', 'barco', 'camión']

In [ ]:
# Obtener un lote de ejemplos y graficarlos
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Desnormalizar para visualizar
def imshow_cifar(img):
    img = img.numpy().transpose((1, 2, 0))
    img = desv * img + media
    return np.clip(img, 0, 1)

display_size = 10
fig = plt.figure(figsize=(25, 4))
for idx in np.arange(display_size):
    ax = fig.add_subplot(2, display_size, idx + 1, xticks=[], yticks=[])
    ax.imshow(imshow_cifar(images[idx]))
    ax.set_title(classes[labels[idx]])

## Dos arquitecturas de la misma profundidad

Ambas redes tienen tres etapas (16, 32 y 64 canales) y **dos bloques por etapa**. La diferencia es solo el atajo:

- **CNN plana:** cada bloque es `conv → BN → ReLU → conv → BN → ReLU`.
- **ResNet:** el mismo bloque, pero se **suma** la entrada (o una proyección $1 \times 1$ si cambian canales o resolución) **antes** del último ReLU.

La CNN plana ya está implementada para que sirva de referencia. Tu trabajo es completar el bloque residual.

In [ ]:
class BloquePlano(nn.Module):
    '''Bloque convolucional sin conexión residual.'''
    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return x

### Bloque residual

Si `stride=1` y no cambian los canales, el atajo es la identidad: sumas $x$ consigo mismo después de $F(x)$.

Si `stride=2` o cambian los canales, $x$ y $F(x)$ no tienen la misma forma. En ese caso usamos una convolución $1 \times 1$ (ya definida en `self.atajo`) para proyectar $x$.

Ejemplo de suma residual:

```
identidad = self.atajo(x)
salida = F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))))
# falta sumar la identidad *antes* del ReLU final
```

El ReLU se aplica **después** de la suma: $H(x) = \mathrm{ReLU}(F(x) + x)$.

In [ ]:
class BloqueResidual(nn.Module):
    '''Bloque residual. Completa el pase frontal.'''
    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)

        # Atajo: identidad, o proyección 1x1 si cambian canales o resolución
        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida)
            )

    def forward(self, x):
        # TODO: Implementa F(x) + atajo(x) y aplica ReLU al resultado (3 puntos)
        # 1. Calcula la transformación F(x) con conv1-bn1-relu y conv2-bn2
        # 2. Suma el atajo: self.atajo(x)
        # 3. Aplica ReLU a la suma y regresa el tensor
        pass

La clase `RedCIFAR` arma las tres etapas. Le pasamos el tipo de bloque (`BloquePlano` o `BloqueResidual`) para que ambas redes tengan la misma profundidad.

In [ ]:
class RedCIFAR(nn.Module):
    def __init__(self, bloque, n_clases=10):
        super().__init__()
        self.conv0 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn0 = nn.BatchNorm2d(16)

        # Dos bloques por etapa: 16, 32 y 64 canales
        self.etapa1 = self._hacer_etapa(bloque, 16, 16, n_bloques=2, stride=1)
        self.etapa2 = self._hacer_etapa(bloque, 16, 32, n_bloques=2, stride=2)
        self.etapa3 = self._hacer_etapa(bloque, 32, 64, n_bloques=2, stride=2)

        self.fc = nn.Linear(64, n_clases)

    def _hacer_etapa(self, bloque, canales_entrada, canales_salida, n_bloques, stride):
        capas = [bloque(canales_entrada, canales_salida, stride)]
        for _ in range(1, n_bloques):
            capas.append(bloque(canales_salida, canales_salida, stride=1))
        return nn.Sequential(*capas)

    def forward(self, x):
        x = F.relu(self.bn0(self.conv0(x)))
        x = self.etapa1(x)
        x = self.etapa2(x)
        x = self.etapa3(x)
        x = F.avg_pool2d(x, 8)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def contar_parametros(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


cnn_plana = RedCIFAR(BloquePlano).to(device)
resnet = RedCIFAR(BloqueResidual).to(device)

print('Parámetros CNN plana: ', contar_parametros(cnn_plana))
print('Parámetros ResNet:    ', contar_parametros(resnet))

## Entrenamiento y validación

Entrenamos las dos redes con los mismos hiperparámetros (optimizador, tasa de aprendizaje, épocas). La función de costo es entropía cruzada sobre los logits.

Si entrenas en CPU puede tardar varios minutos; con GPU es más rápido. Cinco épocas bastan para ver la tendencia; si lo deseas puedes aumentar `epocas`.

In [ ]:
def exactitud(model, dataloader):
    model.eval()
    correctos = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            salidas = model(images)
            _, predichos = torch.max(salidas, 1)
            total += labels.size(0)
            correctos += (predichos == labels).sum().item()
    model.train()
    return correctos / total


def entrenar(model, trainloader, testloader, criterion, optimizer, epocas):
    historial = {'perdida_train': [], 'exactitud_test': []}
    for e in range(epocas):
        model.train()
        running_loss = 0.0
        n_lotes = 0
        t0 = time.time()
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            salidas = model(images)
            loss = criterion(salidas, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_lotes += 1

        perdida_media = running_loss / n_lotes
        acc_test = exactitud(model, testloader)
        historial['perdida_train'].append(perdida_media)
        historial['exactitud_test'].append(acc_test)
        print('Epoch: {}/{}..  Pérdida de entrenamiento: {:.3f}..  Exactitud de prueba: {:.3f}..  Tiempo: {:.1f}s'.format(
            e + 1, epocas, perdida_media, acc_test, time.time() - t0))
    return historial

In [ ]:
criterion = nn.CrossEntropyLoss()

print('--- CNN plana ---')
cnn_plana = RedCIFAR(BloquePlano).to(device)
optim_plana = optim.Adam(cnn_plana.parameters(), lr=tasa_de_aprendizaje)
hist_plana = entrenar(cnn_plana, trainloader, testloader, criterion, optim_plana, epocas)

print('\n--- ResNet ---')
resnet = RedCIFAR(BloqueResidual).to(device)
optim_res = optim.Adam(resnet.parameters(), lr=tasa_de_aprendizaje)
hist_res = entrenar(resnet, trainloader, testloader, criterion, optim_res, epocas)

## Comparación

Grafica las curvas de las dos redes. La residual suele bajar más rápido la pérdida y alcanzar mejor exactitud con la misma profundidad. Si la CNN plana se estanca o queda por debajo, es el efecto de degradación que motivó a ResNet.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_x = np.arange(1, epocas + 1)
axes[0].plot(epochs_x, hist_plana['perdida_train'], label='CNN plana')
axes[0].plot(epochs_x, hist_res['perdida_train'], label='ResNet')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida de entrenamiento')
axes[0].legend()
axes[0].set_title('Pérdida')

axes[1].plot(epochs_x, hist_plana['exactitud_test'], label='CNN plana')
axes[1].plot(epochs_x, hist_res['exactitud_test'], label='ResNet')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Exactitud de prueba')
axes[1].legend()
axes[1].set_title('Exactitud')

plt.tight_layout()

## Actividad

> TODO (1 punto): Escribe una tabla con el resultado de cada red. Las columnas deben ser: arquitectura, pérdida de entrenamiento (última época), exactitud de validación (última época). Comenta en una o dos oraciones el resultado del experimento.